# 🏥 IUI Prediction - Model Training & XAI (ข้อมูลทั้งหมด)

In [1]:
import sys
print(sys.executable)

# ติดตั้ง lightgbm ให้ถูก interpreter ที่ notebook ใช้จริง
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "lightgbm"], check=True)

import os
if os.getcwd().endswith("src"):
    os.chdir("..")

import re
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline

from sklearn.model_selection import GroupShuffleSplit, RandomizedSearchCV, GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, roc_curve, brier_score_loss
)
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression

from imblearn.over_sampling import SMOTE, ADASYN

import shap

warnings.filterwarnings("ignore")


def clean_column_names(df):
    df = df.copy()
    df.columns = [
        re.sub(r"[\[\]<>]", "_", str(col)).replace(" ", "_")
        for col in df.columns
    ]
    return df


def find_optimal_threshold(y_true, y_prob):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    thresholds = np.clip(thresholds, 0, 1)
    j_scores = tpr - fpr
    return float(thresholds[np.argmax(j_scores)])


def calculate_net_benefit(y_true, y_prob, threshold):
    if threshold <= 0 or threshold >= 1:
        return np.nan

    tp = np.logical_and(y_true == 1, y_prob >= threshold).sum()
    fp = np.logical_and(y_true == 0, y_prob >= threshold).sum()
    n = len(y_true)

    return (tp / n) - (fp / n) * (threshold / (1 - threshold))


def plot_confusion_matrix(
    y_true,
    y_pred,
    model_name,
    threshold,
    outdir="reports_temporal/figures/confusion_matrices"
):
    os.makedirs(outdir, exist_ok=True)
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Non-Preg (0)", "Preg (1)"],
        yticklabels=["Non-Preg (0)", "Preg (1)"]
    )
    plt.title(f"Confusion Matrix\n{model_name}\n(Threshold: {threshold:.3f})")
    plt.ylabel("Actual Label")
    plt.xlabel("Predicted Label")
    plt.tight_layout()
    plt.savefig(f"{outdir}/{model_name}_CM.png")
    plt.close()


def plot_roc_curve(
    y_true,
    y_prob,
    model_name,
    auc_score,
    best_threshold,
    outdir="reports_temporal/figures/roc_curves"
):
    os.makedirs(outdir, exist_ok=True)
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    thresholds = np.clip(thresholds, 0, 1)

    best_idx = np.argmin(np.abs(thresholds - best_threshold))
    best_fpr, best_tpr = fpr[best_idx], tpr[best_idx]

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, lw=2, label=f"ROC curve (area = {auc_score:.3f})")
    plt.plot([0, 1], [0, 1], lw=2, linestyle="--")
    plt.scatter(
        [best_fpr],
        [best_tpr],
        marker="o",
        s=100,
        label=f"Optimal Threshold ({best_threshold:.2f})",
        zorder=5
    )
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve: {model_name}")
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(f"{outdir}/{model_name}_ROC.png")
    plt.close()


print("✅ SETUP OK")

c:\Users\HP\AppData\Local\Programs\Python\Python312\python.exe
✅ SETUP OK


c:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 2. โหลดและเตรียมข้อมูล (Standard - ข้อมูลทั้งหมด)

In [2]:
# ==========================================
# Block 2: Load data, temporal split, and impute
# Temporal validation: Train = 2017-2023, Test = 2024-2025
# ==========================================

df = pd.read_csv('data/processed/cycle_level_features_temporal.csv')
df = clean_column_names(df)
df = df.loc[:, ~df.columns.str.startswith('Unnamed')]

# -----------------------------
# Parse Date
# -----------------------------
if 'Date' not in df.columns:
    raise ValueError("Date column not found.")

df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

print("Date missing rate:", df['Date'].isna().mean())
print("Date range:", df['Date'].min(), "to", df['Date'].max())

if df['Date'].isna().all():
    raise ValueError("Date parsing failed: all values are NaN.")

# -----------------------------
# Target cleaning
# -----------------------------
if 'Result' not in df.columns:
    raise ValueError("Target column 'Result' not found.")

df['Result'] = pd.to_numeric(df['Result'], errors='coerce')
df = df.dropna(subset=['Result'])

valid_target = {0, 1, 0.0, 1.0}
bad_target_vals = sorted(set(df['Result'].unique()) - valid_target)
if bad_target_vals:
    raise ValueError(f"Unexpected values in Result: {bad_target_vals}")

df['Result'] = df['Result'].astype(int)

# -----------------------------
# Convert object → numeric
# -----------------------------
object_cols = df.select_dtypes(include=['object']).columns

for c in object_cols:
    if c not in ['HN', 'Result', 'Date']:
        df[c] = pd.to_numeric(df[c], errors='coerce')

print("Remaining object columns:", df.select_dtypes(include=['object']).columns.tolist())

# -----------------------------
# HN cleaning
# -----------------------------
if 'HN' not in df.columns:
    raise ValueError("Column 'HN' is required.")

df = df.dropna(subset=['HN'])
df['HN'] = df['HN'].astype(str).str.strip()
df = df[df['HN'] != ""]

# -----------------------------
# Temporal split
# Train: 2017–2023
# Test:  2024–2025
# -----------------------------
train_mask = df['Date'].dt.year.between(2017, 2023)
test_mask  = df['Date'].dt.year.between(2024, 2025)

df_train = df[train_mask].copy()
df_test  = df[test_mask].copy()

if len(df_train) == 0:
    raise ValueError("Temporal train set is empty.")
if len(df_test) == 0:
    raise ValueError("Temporal test set is empty.")

print(f"\nTemporal split:")
print(f"  Train (2017–2023): {len(df_train)} cycles")
print(f"  Test  (2024–2025): {len(df_test)} cycles")

# -----------------------------
# Patient overlap (allowed)
# -----------------------------
train_hn = set(df_train['HN'])
test_hn  = set(df_test['HN'])
overlap  = train_hn.intersection(test_hn)

print(f"  HN overlap train/test: {len(overlap)}")
if len(overlap) > 0:
    print("  NOTE: Temporal validation allows patient overlap (split by time).")

# -----------------------------
# Drop Date before modeling
# -----------------------------
df_train = df_train.drop(columns=['Date'], errors='ignore')
df_test  = df_test.drop(columns=['Date'], errors='ignore')

# -----------------------------
# Split X / y
# -----------------------------
X_train = df_train.drop(columns=['Result'], errors='ignore')
X_test  = df_test.drop(columns=['Result'], errors='ignore')
y_train = df_train['Result'].copy()
y_test  = df_test['Result'].copy()

# -----------------------------
# Save test info (for analysis)
# -----------------------------
keep_cols = [c for c in ['HN', 'Cycle_Number'] if c in X_test.columns]
test_info = X_test[keep_cols].copy() if keep_cols else pd.DataFrame(index=X_test.index)

# -----------------------------
# Imputation (train ONLY)
# -----------------------------
cols_to_impute = [c for c in X_train.columns if c != 'HN']

# drop all-NaN columns in train
all_nan_cols = [c for c in cols_to_impute if X_train[c].isna().all()]
if all_nan_cols:
    print("Dropping all-NaN columns in train:", all_nan_cols)
    X_train = X_train.drop(columns=all_nan_cols, errors='ignore')
    X_test  = X_test.drop(columns=all_nan_cols, errors='ignore')

cols_to_impute = [c for c in X_train.columns if c != 'HN']

imputer = SimpleImputer(strategy='median')

X_train_num = pd.DataFrame(
    imputer.fit_transform(X_train[cols_to_impute]),
    columns=cols_to_impute,
    index=X_train.index
)

X_test_num = pd.DataFrame(
    imputer.transform(X_test[cols_to_impute]),
    columns=cols_to_impute,
    index=X_test.index
)

# restore HN (for grouping)
X_train_num['HN'] = X_train['HN'].values
X_test_num['HN']  = X_test['HN'].values

# groups for CV (train only)
groups_train = pd.Series(X_train_num['HN'].values, name='HN')

# drop HN before model
X_train = X_train_num.drop(columns=['HN'])
X_test  = X_test_num.drop(columns=['HN'])

# -----------------------------
# Final sanity checks
# -----------------------------
for bad_col in ['Date', 'HN', 'Result']:
    assert bad_col not in X_train.columns, f'{bad_col} should not be in X_train'
    assert bad_col not in X_test.columns,  f'{bad_col} should not be in X_test'

suspicious_cols = [c for c in X_train.columns if 'result' in c.lower() or 'outcome' in c.lower()]
if suspicious_cols:
    raise ValueError(f"Potential leakage columns found: {suspicious_cols}")

# -----------------------------
# Summary
# -----------------------------
print(f'\nEvent rate train: {y_train.mean():.4f}')
print(f'Event rate test:  {y_test.mean():.4f}')

print(f'\nSplit done. Train: {len(X_train)}, Test: {len(X_test)}')
print(f'Number of features: {X_train.shape[1]}')

print("Train positives:", int(y_train.sum()), "out of", len(y_train))
print("Test positives:",  int(y_test.sum()),  "out of", len(y_test))

Date missing rate: 0.0020373514431239388
Date range: 2017-10-02 00:00:00 to 2025-12-29 00:00:00
Remaining object columns: ['HN']

Temporal split:
  Train (2017–2023): 2401 cycles
  Test  (2024–2025): 538 cycles
  HN overlap train/test: 35
  NOTE: Temporal validation allows patient overlap (split by time).

Event rate train: 0.0625
Event rate test:  0.0613

Split done. Train: 2401, Test: 538
Number of features: 63
Train positives: 150 out of 2401
Test positives: 33 out of 538


In [3]:
# ==========================================
# Temporal Validation
# Apply locked final model from test2 to temporal test set (2024-2025)
# ==========================================

import joblib
import os
import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    brier_score_loss, confusion_matrix
)

# --------------------------------
# Load locked final model from test2
# --------------------------------
final_model_temporal = joblib.load(
    "models_test2/final_model/XGBoost_Baseline_final_feature_budget_model.joblib"
)

# Use feature names directly from the saved model
final_features_test2 = list(final_model_temporal.feature_names_in_)

print(f"Features from test2 model: {len(final_features_test2)}")
for i, f in enumerate(final_features_test2, start=1):
    print(f"  {i}. {f}")

# --------------------------------
# Check feature compatibility
# --------------------------------
missing = [f for f in final_features_test2 if f not in X_test.columns]
if missing:
    raise ValueError(f"Missing features in temporal test set: {missing}")

X_test_temporal = X_test[final_features_test2].copy()

# --------------------------------
# Evaluate on temporal test set
# --------------------------------
temporal_probs   = final_model_temporal.predict_proba(X_test_temporal)[:, 1]
temporal_pr_auc  = average_precision_score(y_test, temporal_probs)
temporal_roc_auc = roc_auc_score(y_test, temporal_probs)
temporal_brier   = brier_score_loss(y_test, temporal_probs)

# Fixed threshold carried over from primary development/validation analysis
temporal_threshold = 0.4618

y_pred = (temporal_probs >= temporal_threshold).astype(int)
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
ppv         = tp / (tp + fp) if (tp + fp) > 0 else np.nan
npv         = tn / (tn + fn) if (tn + fn) > 0 else np.nan

print("\n=== Temporal Validation Results ===")
print("Validation type: transported final model from test2")
print("Test period: 2024-2025")
print(f"Test cycles: {len(y_test)}, Positives: {int(y_test.sum())}")

print(f"\nPR-AUC:      {temporal_pr_auc:.4f}")
print(f"ROC-AUC:     {temporal_roc_auc:.4f}")
print(f"Brier:       {temporal_brier:.4f}")

print("\nConfusion matrix:")
print(cm)
print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")
print(f"Sensitivity: {sensitivity:.3f}")
print(f"Specificity: {specificity:.3f}")
print(f"PPV:         {ppv:.3f}")
print(f"NPV:         {npv:.3f}")

# --------------------------------
# Save results
# --------------------------------
os.makedirs("reports_temporal/tables", exist_ok=True)

temporal_summary = pd.DataFrame([{
    "Validation":   "Temporal (2024-2025)",
    "Model":        "Locked XGBoost + Baseline from test2",
    "Num_Features": len(final_features_test2),
    "PR-AUC":       temporal_pr_auc,
    "ROC-AUC":      temporal_roc_auc,
    "Brier":        temporal_brier,
    "Sensitivity":  sensitivity,
    "Specificity":  specificity,
    "PPV":          ppv,
    "NPV":          npv,
    "Threshold":    temporal_threshold
}])

temporal_summary.to_excel(
    "reports_temporal/tables/Temporal_Validation_Results.xlsx",
    index=False
)

print("\n✅ Temporal validation complete")
print("Saved: reports_temporal/tables/Temporal_Validation_Results.xlsx")

Features from test2 model: 16
  1. Uterine_Factors
  2. Total_Female_Pathology
  3. Ovulatory_Factors
  4. Cycle_Day
  5. Post_TPMSC
  6. First_Count
  7. Pre_Count
  8. Gynecological_Surgical_History
  9. Post_Count
  10. Delta_Motile
  11. Age_Female
  12. First_Progressive_Motile
  13. First_Volume
  14. Menstrual_Interval_Days
  15. First_Motile
  16. BMI_InfertilityType_Interaction

=== Temporal Validation Results ===
Validation type: transported final model from test2
Test period: 2024-2025
Test cycles: 538, Positives: 33

PR-AUC:      0.2434
ROC-AUC:     0.7894
Brier:       0.2053

Confusion matrix:
[[233 272]
 [  6  27]]
TN=233, FP=272, FN=6, TP=27
Sensitivity: 0.818
Specificity: 0.461
PPV:         0.090
NPV:         0.975

✅ Temporal validation complete
Saved: reports_temporal/tables/Temporal_Validation_Results.xlsx
